# Criptografía en Solana: Curva Ed25519 (Edwards Retorcidas)

Este notebook explora la matemática detrás de las direcciones y firmas en **Solana**. Mientras que Bitcoin y Ethereum usan curvas de Weierstrass ($y^2 = x^3 + 7$), Solana utiliza **Ed25519**, basada en una curva de Edwards retorcida.

### 1. Preparación del Entorno

In [1]:
# Instalación de librerías para criptografía y visualización interactiva
!pip install pynacl base58 plotly ipywidgets numpy

### 2. Geometría de las Curvas de Edwards

La ecuación de una curva de Edwards retorcida es:
$$ax^2 + y^2 = 1 + dx^2y^2$$

Para **Ed25519**:
- $a = -1$
- $d = -121665/121666$
- Campo primo $p = 2^{255} - 19$

A diferencia de las curvas de Bitcoin, estas curvas suelen ser cerradas y no tienen puntos en el infinito, lo que simplifica y acelera los cálculos.

In [2]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

def plot_edwards_interactive(d):
    # Generamos malla de puntos para la visualización
    limit = 2.5
    res = 200
    
    x = np.linspace(-limit, limit, res)
    y = np.linspace(-limit, limit, res)
    X, Y = np.meshgrid(x, y)
    
    # Ecuación: -x^2 + y^2 = 1 + d * x^2 * y^2
    # Reorganizado para contorno: -x^2 + y^2 - 1 - d*x^2*y^2 = 0
    Z = -X**2 + Y**2 - 1 - d * (X**2) * (Y**2)
    
    fig = go.Figure(data=go.Contour(
        z=Z,
        x=x,
        y=y,
        contours_coloring='lines',
        line_width=3,
        contours=dict(start=0, end=0, size=1),
        line_color='darkorange'
    ))
    
    fig.update_layout(
        title=f"Curva de Edwards Retorcida: -x² + y² = 1 + ({d})x²y²",
        xaxis_title="Eje X",
        yaxis_title="Eje Y",
        width=700,
        height=700,
        xaxis=dict(range=[-limit, limit]),
        yaxis=dict(range=[-limit, limit]),
        template="plotly_dark"
    )
    fig.show()

interact(plot_edwards_interactive, 
         d=FloatSlider(value=-0.9, min=-5.0, max=1, step=0.1, description='Parámetro d'))

interactive(children=(FloatSlider(value=-0.9, description='Parámetro d', max=1.0, min=-5.0), Output()), _dom_c…

<function __main__.plot_edwards_interactive(d)>

### 3. Derivación de Wallets en Solana

En Solana, el proceso es directo:
1. Se genera una **semilla** de 32 bytes (Clave Privada).
2. Se calcula el punto en la curva Ed25519 (Clave Pública).
3. La **dirección** es simplemente la Clave Pública codificada en **Base58**.

*Nota: A diferencia de Ethereum, no hay un paso de hashing adicional (Keccak) para la dirección final.*

In [3]:
import nacl.signing
import base58

def generate_solana_wallet():
    # Crear clave de firma (Privada)
    signing_key = nacl.signing.SigningKey.generate()
    
    # Extraer clave de verificación (Pública)
    verify_key = signing_key.verify_key
    pub_bytes = verify_key.encode()
    
    # Codificar en Base58
    address = base58.b58encode(pub_bytes).decode('utf-8')
    
    print(f"🔑 Private Seed: {signing_key.encode().hex()}")
    print(f"🔓 Public Key (Hex): {pub_bytes.hex()}")
    print(f"☀️ Solana Address: {address}")

generate_solana_wallet()

🔑 Private Seed: 06c8b9258fdcab659810f6a19e5253b76cd3e689065c5ddeafb383b212df5da4
🔓 Public Key (Hex): 216d8887192a6b04c1e75e46a76d6b0e7d50eb1a25094f66610ecd8a2a6cc4d3
☀️ Solana Address: 3FVKu8gwBdDxcv46G2yGiJwS1hBHXRTp6YtuzroLqRdU


### 4. Eficiencia de Firmas (EdDSA)

Solana requiere procesar miles de transacciones por segundo. Ed25519 es ideal porque permite:
- **Firmas deterministas**: No requieren un número aleatorio por firma (evita errores críticos de seguridad).
- **Verificación por lotes**: Se pueden verificar muchas firmas simultáneamente más rápido que de una en una.

Usa la siguiente celda para simular la firma de una transacción:

In [4]:
message = b"Transfer 5.0 SOL to 7v...x4"
key = nacl.signing.SigningKey.generate()

# Firmar el mensaje
signature = key.sign(message)

print(f"Mensaje: {message.decode()}")
print(f"Firma (64 bytes): {signature.signature.hex()[:64]}...")

# Verificar
try:
    key.verify_key.verify(signature)
    print("✅ Verificación exitosa en Solana")
except:
    print("❌ Fallo en la verificación")

Mensaje: Transfer 5.0 SOL to 7v...x4
Firma (64 bytes): 8d9caeb3794336adb35767d1f45749db43690f3f2f361a6f39b11f620568f4a2...
✅ Verificación exitosa en Solana


### 5. Comparativa Visual: Edwards vs Weierstrass

La curva de Edwards (Solana) es un lazo cerrado y suave, mientras que la de Weierstrass (BTC/ETH) es abierta y se extiende al infinito. Esta diferencia topológica es la que permite que las operaciones en Ed25519 sean más rápidas y seguras contra ataques de temporización.